# AI assist proposal demo

This notebook uses a fake Responses API client so it can run without an API key. The AI assist waits for stable input, creates a proposal, and does not apply it until you accept it.

In [1]:
import json
from types import SimpleNamespace

from aipywidgets import AIConfig, AIForm, Action, WhenIdle, fields

In [2]:
class FakeResponses:
    def __init__(self):
        self.calls = 0

    def create(self, **kwargs):
        self.calls += 1
        latest_user_message = next(
            (
                item.get("content", "")
                for item in reversed(kwargs.get("input", []))
                if item.get("role") == "user"
            ),
            "",
        )
        output_path = (
            "full_width_keywords"
            if "Assist id: suggest_keywords_full_width" in latest_user_message
            else "keywords"
        )
        variants = [
            ("Suggested keywords from the abstract.", ["notebook", "metadata", "demo"]),
            ("Adjusted keywords for research data.", ["research-data", "metadata", "review"]),
            ("Adjusted keywords for repository deposit.", ["repository", "dataset", "deposit"]),
            ("Adjusted keywords for curation workflow.", ["curation", "metadata", "workflow"]),
            ("Adjusted keywords for reproducibility.", ["reproducibility", "dataset", "documentation"]),
            ("Adjusted keywords for notebook-based entry.", ["notebook", "data-entry", "review"]),
        ]
        message, keywords = variants[(self.calls - 1) % len(variants)]
        message = f"{message} (proposal {self.calls})"
        proposal = json.dumps(
            {
                "message": message,
                "operations": [
                    {"op": "set", "path": output_path, "value": keywords}
                ],
            }
        )
        return SimpleNamespace(
            output=[
                {
                    "type": "function_call",
                    "name": "propose_form_update",
                    "call_id": f"call_fake_{self.calls}",
                    "arguments": proposal,
                }
            ]
        )


class FakeClient:
    responses = FakeResponses()


In [3]:
# To try OpenAI instead of FakeClient, uncomment these imports and the ai= line below.
# from getpass import getpass
# from openai import OpenAI


form = AIForm(
    title="AI assist demo",
    ai=AIConfig(client=FakeClient(), model="fake-model"),
    # ai=AIConfig(client=OpenAI(api_key=getpass("OpenAI API key: ")), model="gpt-5.4-mini"),
    style={"margin_bottom": "380px"},
    steps=[
        {
            "id": "main",
            "label": "Main",
            "fields": [
                fields.Textarea("abstract", label="Abstract"),
                fields.Tags("keywords", label="Keywords"),
                fields.Textarea("full_width_abstract", label="Abstract (full width)", full_width=True),
                fields.Tags("full_width_keywords", label="Keywords (full width demo)"),
            ],
        }
    ],
    actions=[Action(id="save", label="Save")],
)


form.ai.assist(
    id="suggest_keywords",
    label="Suggest keywords",
    watch=["abstract"],
    trigger=WhenIdle(ms=1200, min_chars=20),
    prompt="Suggest keywords for {{ values.abstract }}",
    outputs={"keywords": "A short list of dataset keywords"},
)

form.ai.assist(
    id="suggest_keywords_full_width",
    label="Suggest keywords for full-width field",
    watch=["full_width_abstract"],
    trigger=WhenIdle(ms=1200, min_chars=20),
    prompt="Suggest keywords for {{ values.full_width_abstract }}",
    outputs={"full_width_keywords": "A short list of dataset keywords"},
)


@form.on_action("save")
def save(ctx):
    ctx.info(f"Saved: {ctx.values}")


form

Edit **Abstract** and pause to verify the standard right-side bubble. Edit **Abstract (full width)** and pause to verify the below-field bubble. The matching **Keywords** field stays unchanged until you press **Accept**.

In [4]:
import io, logging
from aipywidgets import AIConfig, AIForm, Action, WhenIdle, fields

stream = io.StringIO()
handler = logging.StreamHandler(stream)
logger = logging.getLogger("aipywidgets.ai")
old_handlers = logger.handlers[:]
old_level = logger.level
logger.handlers = [handler]
logger.setLevel(logging.WARNING)

class DummyResponses:
    def create(self, **kwargs):
        raise AssertionError("should not call responses.create here")

class DummyClient:
    responses = DummyResponses()

raw_form = AIForm(
    title="Raw",
    ai=AIConfig(client=DummyClient(), model="dummy"),
    steps=[{"id": "main", "label": "Main", "fields": [fields.Textarea("abstract"), fields.Tags("keywords")]}],
    actions=[Action(id="save", label="Save")],
)
raw_form.ai.assist(
    id="suggest_keywords",
    label="Suggest keywords",
    watch=["abstract"],
    trigger=WhenIdle(ms=1200, min_chars=20),
    prompt="Suggest keywords for {{ values.abstract }}",
    outputs={"keywords": "A short list of dataset keywords"},
)
raw_form.set_value("abstract", "This text is long enough to trigger the min_chars guard.")

handler.flush()
output = stream.getvalue()
logger.handlers = old_handlers
logger.setLevel(old_level)
output

'Cannot schedule AI assist before widget() is rendered: suggest_keywords\n'